# ICP × Saúde de Estoque × Curva ABC

Correlaciona a saúde de estoque diária (`sop_gold.stock_health`) com a curva ABC de produtos
(`demand_prediction_input`) e o ICP atual por produto (carteira ativa em `supply_chain_efficiency_model_input`).

**Pergunta central:** variações de ICP representam ameaça real ao estoque? Quão crítico é o produto na curva ABC?

| Seção | Conteúdo |
|---|---|
| 1. Stock Health × ABC × ICP | Snapshot diário enriquecido com tag ABC e ICP atual por produto |
| 2. Histórico ICP por Fornecedor | Tendência de ICP por fornecedor (`scale_ra`, últimos 90 dias) |
| 3. Classificação de Risco | CRITICO / ALTO / MEDIO / MONITORAMENTO / SEM_OP_ATIVA / OK |
| 4. Visualizações | Tabela de alertas + scatter ICP × cobertura + trend de fornecedor |
| 5. Export | CSVs completo e filtrado por alerta |

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import datetime
from pathlib import Path

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="insider-data-lake")

# ----- Parâmetros -----
REF_DATE      = pd.to_datetime("today").normalize()
today         = REF_DATE.strftime("%Y%m%d")
dia_ref       = (REF_DATE - pd.Timedelta(days=1)).strftime("%Y-%m-%d")  # snapshot mais recente

ICP_THRESHOLD = 0.70          # limiar de alerta (ICP abaixo = risco)
ABC_RISCO     = ["A", "B"]    # classes consideradas críticas na curva ABC

output_path = Path("../../outputs/icp_saude_estoque")
output_path.mkdir(parents=True, exist_ok=True)
sql_path    = Path("./sql/stock_health_abc_icp.sql")

print(f"Data de referência : {dia_ref}")
print(f"ICP threshold      : {ICP_THRESHOLD:.0%}")
print(f"ABC risco          : {', '.join(ABC_RISCO)}")

In [ ]:
def query_to_dataframe(query: str) -> pd.DataFrame:
    """Executes a BigQuery SQL query and returns a DataFrame."""
    return client.query(query).result().to_dataframe()


def read_sql_file(path) -> str:
    """Reads a .sql file and returns its content as string."""
    return Path(path).read_text(encoding="utf-8")


fmt_pct = lambda v: f"{v:.1%}" if pd.notna(v) else "—"
fmt_int = lambda v: f"{int(v):,}" if pd.notna(v) else "—"

## 1. Stock Health × ABC × ICP

In [ ]:
def load_stock_health_abc_icp(ref_date: str = dia_ref) -> pd.DataFrame:
    """
    Loads daily stock health enriched with:
    - sku_state and product_name from integrated.skus
    - ABC curve tag from demand_prediction_input (last 3 closed months)
    - Weighted average ICP per product from active OPs (supply_chain_efficiency_model_input)

    Args:
        ref_date: YYYY-MM-DD date to filter stock_health snapshot (grain: sku x dia).

    Returns:
        DataFrame with grain sku x dia. Excludes sku_state='desativado'.
    """
    sql = read_sql_file(sql_path).format(dia_ref=ref_date)
    return query_to_dataframe(sql)


print(f"⏳ Carregando stock health × ABC × ICP  |  dia_ref={dia_ref} ...")
df_main = load_stock_health_abc_icp()
print(f"✅ df_main         : {len(df_main):,} linhas | {df_main.shape[1]} colunas")
print(f"   SKUs distintos   : {df_main['sku'].nunique():,}")
print(f"   Produtos          : {df_main['product_name'].nunique():,}")
print(f"   flag_ameaca=True  : {df_main['flag_ameaca_estoque'].sum():,} SKUs")
df_main.head(3)

## 2. Histórico de ICP por Fornecedor

`scale_ra` tem grão `supplier_name × reference_date` — sem desagregação por produto.  
Usada aqui para identificar **tendência** de ICP dos fornecedores que atendem produtos em risco.

In [ ]:
SQL_ICP_HISTORICO = """
SELECT
    reference_date,
    supplier_name,
    icp,
    total_planned_quantity_icp,
    total_received_quantity_icp
FROM `insider-data-lake.sop_silver.scale_ra`
WHERE DATE(reference_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
ORDER BY reference_date DESC, supplier_name
"""


def load_icp_historico() -> pd.DataFrame:
    """
    Loads historical ICP per supplier from scale_ra (last 90 days).
    Use to identify suppliers with declining ICP trend.

    Returns:
        DataFrame with grain: supplier_name x reference_date.
        icp = total_received_quantity_icp / total_planned_quantity_icp
    """
    df = query_to_dataframe(SQL_ICP_HISTORICO)
    df["reference_date"] = pd.to_datetime(df["reference_date"])
    return df


print("⏳ Carregando histórico de ICP por fornecedor...")
df_icp_hist = load_icp_historico()
print(f"✅ df_icp_hist  : {len(df_icp_hist):,} linhas | {df_icp_hist['supplier_name'].nunique()} fornecedores")
df_icp_hist.head(3)

## 3. Classificação de Risco

In [ ]:
def classify_risk(df: pd.DataFrame, icp_threshold: float = ICP_THRESHOLD) -> pd.DataFrame:
    """
    Classifies stock health risk combining ICP level with ABC tag.
    Overrides the SQL-computed nivel_risco with the parametrizable threshold.

    Levels (in increasing severity):
        OK           : ICP >= threshold, tag C ou sem tag
        MONITORAMENTO: ICP >= threshold, tag A ou B
        MEDIO        : ICP < threshold, tag C
        ALTO         : ICP < threshold, tag B
        CRITICO      : ICP < threshold, tag A
        SEM_OP_ATIVA : ICP = NaN (sem OP ativa na carteira), tag A ou B

    Args:
        df: DataFrame com colunas avg_icp e tag_abc (saída de load_stock_health_abc_icp).
        icp_threshold: Limiar de ICP para classificar risco. Default: ICP_THRESHOLD global.

    Returns:
        DataFrame com colunas nivel_risco e flag_ameaca_estoque atualizadas.
    """
    df = df.copy()

    cond_icp_baixo = df["avg_icp"].notna() & (df["avg_icp"] < icp_threshold)
    cond_icp_ok    = df["avg_icp"].notna() & (df["avg_icp"] >= icp_threshold)
    cond_sem_icp   = df["avg_icp"].isna()
    cond_ab        = df["tag_abc"].isin(["A", "B"])
    cond_a         = df["tag_abc"] == "A"
    cond_b         = df["tag_abc"] == "B"
    cond_c         = df["tag_abc"] == "C"

    # Atribuição em ordem crescente de severidade (última regra vence)
    df["nivel_risco"] = "OK"
    df.loc[cond_icp_baixo & cond_c,  "nivel_risco"] = "MEDIO"
    df.loc[cond_icp_ok    & cond_ab, "nivel_risco"] = "MONITORAMENTO"
    df.loc[cond_sem_icp   & cond_ab, "nivel_risco"] = "SEM_OP_ATIVA"
    df.loc[cond_icp_baixo & cond_b,  "nivel_risco"] = "ALTO"
    df.loc[cond_icp_baixo & cond_a,  "nivel_risco"] = "CRITICO"

    df["flag_ameaca_estoque"] = cond_icp_baixo & cond_ab
    return df


df_main = classify_risk(df_main)

ORDEM_RISCO = {"CRITICO": 0, "ALTO": 1, "SEM_OP_ATIVA": 2, "MEDIO": 3, "MONITORAMENTO": 4, "OK": 5}

resumo = (
    df_main.drop_duplicates("sku")
    .groupby("nivel_risco", observed=True)
    .agg(
        n_skus     = ("sku",          "count"),
        n_produtos = ("product_name", "nunique"),
        icp_medio  = ("avg_icp",      "mean"),
    )
    .reset_index()
    .sort_values("nivel_risco", key=lambda s: s.map(ORDEM_RISCO))
)
resumo["icp_medio"] = resumo["icp_medio"].apply(fmt_pct)

print("📊 Resumo por nível de risco:")
display(resumo)

## 4. Visualizações

In [ ]:
CORES_RISCO = {
    "CRITICO":       "#DC2626",
    "ALTO":          "#F97316",
    "SEM_OP_ATIVA":  "#A78BFA",
    "MEDIO":         "#FBBF24",
    "MONITORAMENTO": "#60A5FA",
    "OK":            "#6EE7B7",
}

# Tabela de alertas — 1 linha por produto (dedup), ordenada por severidade
alertas = (
    df_main[df_main["nivel_risco"].isin(["CRITICO", "ALTO", "SEM_OP_ATIVA"])]
    .drop_duplicates("product_name")
    .assign(
        icp_fmt       = lambda r: r["avg_icp"].apply(fmt_pct),
        cobertura_fmt = lambda r: r["estoque_passado_ou_projetado_d"].apply(
            lambda v: f"{v:.0f} d" if pd.notna(v) else "—"
        ),
    )
    .sort_values("nivel_risco", key=lambda s: s.map(ORDEM_RISCO))
    [["product_name", "tag_abc", "icp_fmt", "nivel_risco",
      "cobertura_fmt", "stock_classification", "fornecedores"]]
    .rename(columns={
        "product_name":         "Produto",
        "tag_abc":              "ABC",
        "icp_fmt":              "ICP",
        "nivel_risco":          "Risco",
        "cobertura_fmt":        "Cobertura",
        "stock_classification": "Stock Class",
        "fornecedores":         "Fornecedores",
    })
    .reset_index(drop=True)
)

print(f"🚨 Produtos em CRITICO / ALTO / SEM_OP_ATIVA: {len(alertas)}")
display(alertas)

In [ ]:
# Scatter: ICP médio × cobertura de estoque — 1 ponto por produto
scatter_data = (
    df_main[df_main["avg_icp"].notna()]
    .groupby("product_name")
    .agg(
        tag_abc         = ("tag_abc",                       "first"),
        nivel_risco     = ("nivel_risco",                   "first"),
        avg_icp         = ("avg_icp",                       "first"),
        cobertura_media = ("estoque_passado_ou_projetado_d", "median"),
        n_skus          = ("sku",                           "nunique"),
        fornecedores    = ("fornecedores",                  "first"),
    )
    .reset_index()
)

fig_scatter = px.scatter(
    scatter_data,
    x="avg_icp",
    y="cobertura_media",
    color="nivel_risco",
    color_discrete_map=CORES_RISCO,
    symbol="tag_abc",
    size="n_skus",
    size_max=20,
    hover_name="product_name",
    hover_data={
        "tag_abc":         True,
        "avg_icp":         ":.1%",
        "cobertura_media": ":.0f",
        "fornecedores":    True,
        "n_skus":          True,
    },
    labels={
        "avg_icp":         "ICP médio (produto)",
        "cobertura_media": "Cobertura mediana (dias)",
        "nivel_risco":     "Nível de Risco",
        "tag_abc":         "ABC",
    },
    title=f"ICP × Cobertura de Estoque por Produto  |  dia_ref={dia_ref}",
)
fig_scatter.add_vline(
    x=ICP_THRESHOLD, line_dash="dash", line_color="#6B7280",
    annotation_text=f"Threshold ICP ({ICP_THRESHOLD:.0%})",
    annotation_position="top right",
)
fig_scatter.update_layout(height=540, template="plotly_white", xaxis_tickformat=".0%")
fig_scatter.show()

In [ ]:
# Tendência ICP — fornecedores dos produtos CRITICO / ALTO
top_fornecedores = (
    df_main[df_main["nivel_risco"].isin(["CRITICO", "ALTO"])]
    .drop_duplicates("product_name")["fornecedores"]
    .dropna()
    .str.split(" | ")
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
    .index.tolist()
)

df_trend = (
    df_icp_hist[df_icp_hist["supplier_name"].isin(top_fornecedores)]
    .sort_values("reference_date")
)

if df_trend.empty:
    print("ℹ️  Nenhum fornecedor CRITICO/ALTO encontrado em scale_ra nos últimos 90 dias.")
else:
    fig_trend = px.line(
        df_trend,
        x="reference_date",
        y="icp",
        color="supplier_name",
        markers=True,
        labels={
            "icp":            "ICP",
            "reference_date": "Data",
            "supplier_name":  "Fornecedor",
        },
        title="Tendência de ICP — Fornecedores de Produtos em Risco (últimos 90 dias)",
    )
    fig_trend.add_hline(
        y=ICP_THRESHOLD, line_dash="dash", line_color="#6B7280",
        annotation_text=f"Threshold ({ICP_THRESHOLD:.0%})",
        annotation_position="right",
    )
    fig_trend.update_layout(height=450, template="plotly_white", yaxis_tickformat=".0%")
    fig_trend.show()

## 5. Export

In [ ]:
# Export completo (todos os SKUs do snapshot)
out_main = output_path / f"icp_abc_correlacao_{today}.csv"
df_main.to_csv(out_main, index=False)
print(f"✅ Completo : {out_main}")
print(f"   {len(df_main):,} linhas | {df_main.shape[1]} colunas")

# Export de alertas (CRITICO + ALTO + SEM_OP_ATIVA)
df_alertas = df_main[df_main["nivel_risco"].isin(["CRITICO", "ALTO", "SEM_OP_ATIVA"])]
out_alertas = output_path / f"alertas_risco_{today}.csv"
df_alertas.to_csv(out_alertas, index=False)
print(f"✅ Alertas  : {out_alertas}")
print(f"   {len(df_alertas):,} linhas | {df_alertas['product_name'].nunique()} produtos em risco")